In [1]:
import carla
import time
import math
from math import *
import random
import CarlaCompatible
import cv2
import Camera
Lane_Detection = CarlaCompatible.LaneDetectorSlidingWindow()
client = carla.Client('localhost', 2000)
world = client.get_world()
map = world.get_map()
spawn_point = map.get_spawn_points()[0]


In [ ]:
world.get_map().name

In [2]:
spectator = world.get_spectator()
spectator.set_transform(spawn_point)

In [ ]:
print(spectator.get_transform())

In [ ]:
spawn_point = spectator.get_transform()

In [4]:
spawn_point = carla.Transform(carla.Location(x=225.234955, y=-374.419922, z=0.267963),carla.Rotation(pitch=0.000000, yaw=-0.295319, roll=0.000000))


In [5]:

vehicle_blueprint= world.get_blueprint_library().filter("*.nissan.*")

vehicle = world.try_spawn_actor(vehicle_blueprint[0], spawn_point)

if vehicle == None:
    
    vehicle = world.get_actors().filter("vehicle.*")
    for v in vehicle:
        v.destroy()

world.wait_for_tick()

vehicle_position = vehicle.get_transform()

camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
camera_bp.set_attribute('sensor_tick', '0.1')

camera_transform = carla.Transform(
    carla.Location(x=1.5, z=1.8),
    carla.Rotation(pitch=-5)
)

camera = Camera.Camera(world,vehicle)


In [ ]:
print(vehicle.get_transform())

In [6]:
def PID_speed(vehicle : carla.Vehicle, target_speed : float, previous_error : float ,previous_int : float, delta_time : float):
    kp = 0.2
    ki = 0.02
    kd = 0.02
    brake = 0.0
    current_speed = vehicle.get_velocity().length()
    error = target_speed - current_speed
    prop_term = kp * error
    int_term = (previous_int + error * delta_time)
    int_term_ki = int_term * ki
    deriv_term = kd * (error - previous_error) / delta_time if delta_time > 0 else 0
    control_signal = prop_term + int_term_ki + deriv_term
    
    control_signal = max(-1 ,min(control_signal,1))

    if control_signal < 0:
        brake = -1 * control_signal
        control_signal = 0

    return control_signal, brake, error, int_term

In [7]:
def steering_angle(Current_position : carla.Transform, Goal : carla.Vector2D):
    # Relative vector from Car to Goal
    # relative_vector = carla.Vector2D(Goal.location.x - Current_position.location.x, Goal.location.y - Current_position.location.y)    
    # # Rotation Matrix
    # Rotation_matrix = [[cos(radians(-Current_position.rotation.yaw)), -sin(radians(-Current_position.rotation.yaw))],
    #                    [sin(radians(-Current_position.rotation.yaw)), cos(radians(-Current_position.rotation.yaw))]]
    
    # # Rotated vector in order with Car
    # rotated_relative_vector = carla.Vector2D(Rotation_matrix[0][0] * relative_vector.x + Rotation_matrix[0][1] * relative_vector.y,
    #                                         Rotation_matrix[1][0] * relative_vector.x + Rotation_matrix[1][1] * relative_vector.y)
    rotated_relative_vector = carla.Vector2D(Goal.x,Goal.y)

    Ld2 = rotated_relative_vector.squared_length()
    curvature = (2 * rotated_relative_vector.y) / Ld2
    angle = degrees(atan(3 * curvature))
    
    
    print(f"rotated_relative_vector: {rotated_relative_vector}")
    print(f"rotated_relative_vector.y: {rotated_relative_vector.y}")
    print(f"curvature: {curvature}")

    return angle

def Pure_pursuit(vehicle : carla.Vehicle, Goal : carla.Transform):

    Next_goal = Goal
    Current_position = vehicle.get_transform()

    Angle = steering_angle(Current_position, Next_goal)
    
    return Angle

In [8]:
def Calc_Adaptive_Speed(P1 : carla.Location, P2 : carla.Location, P3 : carla.Location, Max_speed : float = 20):
    v1 = carla.Vector2D(P2.x - P1.x, P2.y - P1.y)
    v2 = carla.Vector2D(P3.x - P2.x, P3.y - P2.y)
    dot = v1.x * v2.x + v1.y * v2.y
    denom = v1.length() * v2.length()

    cos_theta = dot / denom
    cos_theta = max(-1.0, min(1.0, cos_theta))

    Theta = max(1,degrees(acos(cos_theta))*0.05)
    print(f"theta = {Theta}")
    print(f"Adjusted speed = {1/(Theta) * Max_speed}")
    
    return 1/(Theta) * Max_speed
    

In [9]:
def Make_Path(vehicle : carla.Vehicle, length : int, lookahead : float):
    Path = [map.get_waypoint(vehicle.get_transform().location)]
    while len(Path) < length:
        Next_waypoints = Path[len(Path)-1].next(lookahead)
        Path.append(Next_waypoints[random.randint(0,len(Next_waypoints)-1)])
    return Path



In [ ]:
vehicle.apply_control(carla.VehicleControl(brake=1.0, steer=0))


In [10]:
camera.start_camera()

In [ ]:
camera.stop_camera()

In [ ]:
camera.save_frame()

In [11]:
# Reset Vehicle

list_of_throttle = []
vehicle.set_transform(spawn_point)
vehicle.apply_control(carla.VehicleControl(throttle=0.0, steer=0,brake=0.0))
world.wait_for_tick()
print(vehicle.get_wheel_steer_angle(carla.VehicleWheelLocation.FL_Wheel))
print(vehicle.get_wheel_steer_angle(carla.VehicleWheelLocation.FR_Wheel))


0.0
0.0


In [12]:
# Initializing Parameters for PID controller
error = 0
integral = 0
delta_time = 0
OverAll_time = 0
Max_speed = 10
LookAhead = 7
waypoints = 10
# Path = Make_Path(vehicle,LookAhead,waypoints)
new_point = carla.Vector2D(LookAhead,-Lane_Detection.process_frame(camera.get_current_frame()))


# Drawing Debug Point
# world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(0,255, 0), life_time=10.0)

while True: 


    vehicle_transform = vehicle.get_transform()

    yaw = math.radians(vehicle_transform.rotation.yaw)

    # 1 meter behind the car
    offset_x = -3.0 * math.cos(yaw)
    offset_y = -3.0 * math.sin(yaw)

    spectator_location = vehicle_transform.location + carla.Location(
        x=offset_x,
        y=offset_y,
        z=2.5
    )

    spectator_rotation = vehicle_transform.rotation

    world.get_spectator().set_transform(
        carla.Transform(
            spectator_location,
            spectator_rotation
        )
    )
        
    if Lane_Detection.latest_mask is not None:
        cv2.imshow("mask", Lane_Detection.latest_mask)

    if Lane_Detection.latest_result is not None:
        cv2.imshow("result", Lane_Detection.latest_result)

    if Lane_Detection.latest_sliding_window is not None:
        cv2.imshow("Sliding window", Lane_Detection.latest_sliding_window)
        
    if cv2.waitKey(1) == 27:
        break

    # Euclidean distance between the vehicle and the goal point
    dx = new_point.x 
    dy = new_point.y
    dist = (dx*dx + dy*dy) ** 0.5
    
    new_point = carla.Vector2D(LookAhead,-Lane_Detection.process_frame(camera.get_current_frame()))

    if dist < 2:
        # world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(255, 0, 0), life_time=10.0)
        
        
        new_point = carla.Vector2D(LookAhead,-Lane_Detection(camera.get_current_frame()))

        # world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(0,255, 0), life_time=10.0)
    
    
    # Target_speed = Calc_Adaptive_Speed(vehicle.get_location(),Path[0].transform.location,Path[1].transform.location,Max_speed)
    Target_speed = Max_speed
    
    # Steering Control using Pure Pursuit
    Steerangle = Pure_pursuit(vehicle, new_point) * 0.5
    Angle = Steerangle / 67
    
    
    
    # Throttle Control using PID Controller
    throttle, brake, error, integral = PID_speed(vehicle, Target_speed, error, integral, delta_time)
    

    vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=Angle, brake=brake))

    
    # Debug Information
    print(f"Vehicle Location: {vehicle.get_transform().location}")
    print(f"Goal_Point: {new_point}")
    print(f"required Steering wheel angle: {Steerangle}")
    print(f"Throttle: {throttle}")
    current_speed = vehicle.get_velocity().length()
    list_of_throttle.append([current_speed,throttle - brake ,Steerangle,OverAll_time])
    
    
    snap_shot = world.wait_for_tick()
    delta_time = snap_shot.timestamp.delta_seconds
    OverAll_time += delta_time



rotated_relative_vector: Vector2D(x=7.000000, y=-0.182109)
rotated_relative_vector.y: -0.18210937082767487
curvature: -0.00742800799863853
Vehicle Location: Location(x=225.234955, y=-374.419922, z=-0.018535)
Goal_Point: Vector2D(x=7.000000, y=-0.182109)
required Steering wheel angle: -0.6382846241117922
Throttle: 1
rotated_relative_vector: Vector2D(x=7.000000, y=-0.182109)
rotated_relative_vector.y: -0.18210937082767487
curvature: -0.00742800799863853
Vehicle Location: Location(x=225.234955, y=-374.419922, z=-0.019012)
Goal_Point: Vector2D(x=7.000000, y=-0.182109)
required Steering wheel angle: -0.6382846241117922
Throttle: 1
rotated_relative_vector: Vector2D(x=7.000000, y=-0.182109)
rotated_relative_vector.y: -0.18210937082767487
curvature: -0.00742800799863853
Vehicle Location: Location(x=225.234955, y=-374.419922, z=-0.019168)
Goal_Point: Vector2D(x=7.000000, y=-0.182109)
required Steering wheel angle: -0.6382846241117922
Throttle: 1
rotated_relative_vector: Vector2D(x=7.000000, y=-

c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:158: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit([p[1] for p in left_points], [p[0] for p in left_points], 2)
c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:159: RankWarning: Polyfit may be poorly conditioned
  right_fit = np.polyfit([p[1] for p in right_points], [p[0] for p in right_points], 2)


rotated_relative_vector: Vector2D(x=7.000000, y=0.161875)
rotated_relative_vector.y: 0.16187499463558197
curvature: 0.006603611302392443
Vehicle Location: Location(x=392.818817, y=-183.387802, z=-0.019058)
Goal_Point: Vector2D(x=7.000000, y=0.161875)
required Steering wheel angle: 0.5674643561099794
Throttle: 0.6326429736697303
rotated_relative_vector: Vector2D(x=7.000000, y=0.182109)
rotated_relative_vector.y: 0.18210937082767487
curvature: 0.00742800799863853
Vehicle Location: Location(x=392.788208, y=-182.059708, z=-0.019058)
Goal_Point: Vector2D(x=7.000000, y=0.182109)
required Steering wheel angle: 0.6382846241117922
Throttle: 0.6332757493309827


c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:158: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit([p[1] for p in left_points], [p[0] for p in left_points], 2)
c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:159: RankWarning: Polyfit may be poorly conditioned
  right_fit = np.polyfit([p[1] for p in right_points], [p[0] for p in right_points], 2)


rotated_relative_vector: Vector2D(x=7.000000, y=0.182109)
rotated_relative_vector.y: 0.18210937082767487
curvature: 0.00742800799863853
Vehicle Location: Location(x=392.746552, y=-180.574722, z=-0.019058)
Goal_Point: Vector2D(x=7.000000, y=0.182109)
required Steering wheel angle: 0.6382846241117922
Throttle: 0.6329566863106639
rotated_relative_vector: Vector2D(x=7.000000, y=0.190781)
rotated_relative_vector.y: 0.19078125059604645
curvature: 0.007781210123701272
Vehicle Location: Location(x=392.677673, y=-178.582520, z=-0.019059)
Goal_Point: Vector2D(x=7.000000, y=0.190781)
required Steering wheel angle: 0.6686243169792115
Throttle: 0.6320723205818606


...

c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:158: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit([p[1] for p in left_points], [p[0] for p in left_points], 2)
c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:159: RankWarning: Polyfit may be poorly conditioned
  right_fit = np.polyfit([p[1] for p in right_points], [p[0] for p in right_points], 2)


rotated_relative_vector: Vector2D(x=7.000000, y=0.797813)
rotated_relative_vector.y: 0.7978125214576721
curvature: 0.03214620043925904
Vehicle Location: Location(x=374.151123, y=8.543465, z=-0.008527)
Goal_Point: Vector2D(x=7.000000, y=0.797813)
required Steering wheel angle: 2.754244962627128
Throttle: 0.6331954287367026
rotated_relative_vector: Vector2D(x=7.000000, y=0.690859)
rotated_relative_vector.y: 0.6908593773841858
curvature: 0.02792632398378002
Vehicle Location: Location(x=373.057404, y=9.456264, z=-0.007991)
Goal_Point: Vector2D(x=7.000000, y=0.690859)
required Steering wheel angle: 2.3944989374031067
Throttle: 0.634087971051863
rotated_relative_vector: Vector2D(x=7.000000, y=0.849844)
rotated_relative_vector.y: 0.8498437404632568
curvature: 0.034183651549344295
Vehicle Location: Location(x=371.689667, y=10.541201, z=-0.007162)
Goal_Point: Vector2D(x=7.000000, y=0.849844)
required Steering wheel angle: 2.9276340531564595
Throttle: 0.6338707937917171
rotated_relative_vector: 

c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:158: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit([p[1] for p in left_points], [p[0] for p in left_points], 2)
c:\Users\SOOQ ELASER\Documents\Carla\PythonAPI\Autonomous project\CarlaCompatible.py:159: RankWarning: Polyfit may be poorly conditioned
  right_fit = np.polyfit([p[1] for p in right_points], [p[0] for p in right_points], 2)


rotated_relative_vector: Vector2D(x=7.000000, y=1.286328)
rotated_relative_vector.y: 1.2863280773162842
curvature: 0.050788163623095746
Vehicle Location: Location(x=356.565277, y=18.577301, z=0.181579)
Goal_Point: Vector2D(x=7.000000, y=1.286328)
required Steering wheel angle: 4.331606829600722
Throttle: 0.6454485338328613
rotated_relative_vector: Vector2D(x=7.000000, y=1.127344)
rotated_relative_vector.y: 1.127343773841858
curvature: 0.04485074554097459
Vehicle Location: Location(x=353.570984, y=19.345278, z=0.223196)
Goal_Point: Vector2D(x=7.000000, y=1.127344)
required Steering wheel angle: 3.831625258322715
Throttle: 0.6439238767187901
rotated_relative_vector: Vector2D(x=7.000000, y=0.913437)
rotated_relative_vector.y: 0.9134374856948853
curvature: 0.036658936069121184
Vehicle Location: Location(x=351.697296, y=19.701799, z=0.249138)
Goal_Point: Vector2D(x=7.000000, y=0.913437)
required Steering wheel angle: 3.1379927903048848
Throttle: 0.6424787732324826
rotated_relative_vector: V

KeyboardInterrupt: 

In [ ]:
print(list_of_throttle[-1])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Data format:
# [speed, throttle, steering, time]

times      = [x[3] for x in list_of_throttle]
speeds     = [x[0] for x in list_of_throttle]
throttles  = [x[1] for x in list_of_throttle]
steerings  = [x[2] for x in list_of_throttle]

fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 2, figure=fig)

# ============================================================
# Steering + Throttle
# ============================================================
ax1 = fig.add_subplot(gs[0, 0])

ax1.plot(times, steerings, 'g', label='Steering')
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Steering", color='g')
ax1.tick_params(axis='y', labelcolor='g')
ax1.grid(True)

ax1_t = ax1.twinx()
ax1_t.plot(times, throttles, 'r', label='Throttle')
ax1_t.set_ylabel("Throttle", color='r')
ax1_t.tick_params(axis='y', labelcolor='r')

ax1.set_title("Steering and Throttle vs Time")

# ============================================================
# Speed + Throttle
# ============================================================
ax2 = fig.add_subplot(gs[0, 1])

ax2.plot(times, speeds, 'b', label='Speed')
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Speed (m/s)", color='b')
ax2.tick_params(axis='y', labelcolor='b')
ax2.grid(True)

ax2_t = ax2.twinx()
ax2_t.plot(times, throttles, 'r', label='Throttle')
ax2_t.set_ylabel("Throttle", color='r')
ax2_t.tick_params(axis='y', labelcolor='r')

ax2.set_title("Speed and Throttle vs Time")

# ============================================================
# Speed + Steering
# ============================================================
ax3 = fig.add_subplot(gs[1, :])

ax3.plot(times, speeds, 'b', label='Speed')
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("Speed (m/s)", color='b')
ax3.tick_params(axis='y', labelcolor='b')
ax3.grid(True)

ax3_t = ax3.twinx()
ax3_t.plot(times, steerings, 'g', label='Steering')
ax3_t.set_ylabel("Steering", color='g')
ax3_t.tick_params(axis='y', labelcolor='g')

ax3.set_title("Speed and Steering vs Time")

fig.suptitle("Pure Pursuit + PID Performance", fontsize=16)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
Vehicles = world.get_actors().filter("vehicle.*")
for v in Vehicles:
    v.destroy()
Cameras = world.get_actors().filter("sensor.camera.rgb")
for v in Cameras:
    v.destroy()

In [ ]:

topology = world.get_map().get_topology()
for segment in topology:
    Location = carla.Location(segment[0].transform.location.x, segment[0].transform.location.y, segment[0].transform.location.z + 0.1)
    world.debug.draw_point(Location, size=0.3, color=carla.Color(255, 0, 0))
    time.sleep(5)